In [1]:
import sys, os, glob, time, socket
from loguru import logger
from tabulate import tabulate
import pandas as pd
import numpy as np
import joblib   
import random

In [2]:
ANTIGENS = [
    "Diphtheria",
    "Pertussis",
    "Polio",
    "Tetanus",
    "Rotavirus",
    "PCV",
    "Measles",
    "Mumps",
    "Rubella",
    "Hepatitis_B",
    "Hib",
    "HPV",
]

N_YEARS = 10
YEARS = np.arange(1, N_YEARS + 1)

# Reformat the capacity scenarios

In [3]:
# capacity_scenario_names = [
#     "base_capacity",
#     "pandemic",
# ]
# capacity_scenario_probs = {
#     "base_capacity": 0.8418,
#     "pandemic": 0.0275,
# }

capacity_scenario_names = [
    "base_capacity",
    "IPV_Shortage",
    "pandemic",
    "funding_delay",
    "innacurate_forecast",
    "supply_chain",
    "other",
]
capacity_scenario_probs = {
    "base_capacity": 0.8418,
    "IPV_Shortage": 0.0173,
    "pandemic": 0.0275,
    "funding_delay": 0.0778,
    "innacurate_forecast": 0.0023,
    "supply_chain": 0.0058,
    "other": 0.0275,
}

In [5]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"data\production_capacity_scenarios_v2.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

print(temp)
# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data\production_capacity_scenarios2.csv", index=False)

       Manufacturer  ...  capacity_scenario_probability
0       AJ_Vaccines  ...                         0.0275
1          BB_NCIPD  ...                         0.0275
2    Bharat_Biotech  ...                         0.0275
3         Bilthoven  ...                         0.0275
4      Biological_E  ...                         0.0275
5    China_National  ...                         0.0275
6               GSK  ...                         0.0275
7      Haffkine_Bio  ...                         0.0275
8           LG_Chem  ...                         0.0275
9       Merck_Sharp  ...                         0.0275
10           PT_Bio  ...                         0.0275
11   Panacea_Biotec  ...                         0.0275
12           Pfizer  ...                         0.0275
13           Sanofi  ...                         0.0275
14  Serum_Institute  ...                         0.0275

[15 rows x 13 columns]


In [6]:
capacity_scenarios

,manufacturer,capacity_scenario,capacity_scenario_probability,capacity
0,AJ_Vaccines,base_capacity,0.8418,"[7620215.704973192, 7802935.587913055, 7851893..."
1,BB_NCIPD,base_capacity,0.8418,"[39759132.94554055, 39660680.88477277, 3871949..."
2,Bharat_Biotech,base_capacity,0.8418,"[61532479.59635442, 61358368.2218503, 61420741..."
3,Bilthoven,base_capacity,0.8418,"[11903110.93922455, 12006526.93667252, 1210823..."
4,Biological_E,base_capacity,0.8418,"[163807666.8919225, 162808827.6898424, 1642179..."
...,...,...,...,...
100,PT_Bio,other,0.0275,"[46606392.98975441, 46293481.55039343, 4540880..."
101,Panacea_Biotec,other,0.0275,"[10767253.25728967, 10711528.39392783, 1106849..."
102,Pfizer,other,0.0275,"[85444251.2303147, 84318740.55897625, 83519126..."
103,Sanofi,other,0.0275,"[88590563.9429312, 86836129.89446063, 82910332..."


In [8]:
import pandas as pd
import numpy as np

# Load data
demand_scenarios_df = pd.read_csv(
    "data/OOB/antigen_demand_80_20_5_scenarios2.csv",
    converters={"demands": pd.eval},
    usecols=["demands", "antigen", "demand_SID", "prob"],
)

# capacity_scenarios_df = pd.read_csv(
#     "data/production_capacity_scenarios.csv",
#     converters={"capacity": pd.eval},
# )

capacity_scenarios_df = capacity_scenarios

# Function to generate pairs
def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 10, verbose: bool = True):
    demand_dict = demand.drop_duplicates(subset=["demand_SID"]).set_index("demand_SID")["prob"].to_dict()
    capacity_dict = capacity.drop_duplicates(subset=["capacity_scenario"]).set_index("capacity_scenario")["capacity_scenario_probability"].to_dict()

    demand_keys = list(demand_dict.keys())
    demand_probs = list(demand_dict.values())

    capacity_keys = list(capacity_dict.keys())
    capacity_probs = list(capacity_dict.values())

    if verbose:
        print(f"Unique demand scenarios: {demand_keys}")
        print(f"Unique capacity scenarios: {capacity_keys}")

    pairs = []
    pair_dfs = []
    prob_dict = {}
    pair_idx = 1

    for selected_capacity, selected_prob_capacity in capacity_dict.items():
        available_demand_keys = demand_keys.copy()
        available_demand_probs = demand_probs.copy()

        for _ in range(n_pairs):
            # Normalize the probabilities
            available_demand_probs = [p / sum(available_demand_probs) for p in available_demand_probs]

            selected_demand = np.random.choice(available_demand_keys, p=available_demand_probs)
            # Combine the probabilities
            selected_prob_demand = demand_dict[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_SID"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["prob", "demand_SID"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demands": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)

            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

            # Remove the selected demand from the available demands
            index = available_demand_keys.index(selected_demand)
            available_demand_keys.pop(index)
            available_demand_probs.pop(index)

    pair_df = pd.concat(pair_dfs, ignore_index=True)

    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())

    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)

    return pairs, pair_df

# Generate pairs
scenario_pairs, pair_df = generate_pairs(demand_scenarios_df, capacity_scenarios_df, n_pairs=5, verbose=True)

# Export to json and csv
pair_df.to_json("data/OOB/pair_demand_capacity_new2.json", orient="records", lines=True)
pair_df.to_csv("data/OOB/pair_demand_capacity_new2.csv", index=False)

# Display the DataFrame
pair_df


Unique demand scenarios: [2, 3, 4, 5, 6]
Unique capacity scenarios: ['base_capacity', 'IPV_Shortage', 'pandemic', 'funding_delay', 'innacurate_forecast', 'supply_chain', 'other']


,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[657354000, 751729500, 827191800, 881049800, 8...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
1,HPV,"[27340000, 35973200, 41902900, 49597700, 54454...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
2,Hepatitis_B,"[386971200, 341644400, 323790700, 394259000, 3...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
3,Hib,"[304354400, 308656200, 347768600, 332411700, 3...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
4,Measles,"[429846900, 502939800, 572604400, 617561300, 6...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
...,...,...,...,...,...,...
940,PT_Bio,"[46606392.98975441, 46293481.55039343, 4540880...",manufacturer,35,Demand: 4 - Capacity: other,0.003465
941,Panacea_Biotec,"[10767253.25728967, 10711528.39392783, 1106849...",manufacturer,35,Demand: 4 - Capacity: other,0.003465
942,Pfizer,"[85444251.2303147, 84318740.55897625, 83519126...",manufacturer,35,Demand: 4 - Capacity: other,0.003465
943,Sanofi,"[88590563.9429312, 86836129.89446063, 82910332...",manufacturer,35,Demand: 4 - Capacity: other,0.003465


# Create scenario pairs

In [9]:
pair_df.set_index("pair_idx")

,unit,values,type,pair,pair_probability
pair_idx,,,,,
1,Diphtheria,"[657354000, 751729500, 827191800, 881049800, 8...",antigen,Demand: 4 - Capacity: base_capacity,0.106067
1,HPV,"[27340000, 35973200, 41902900, 49597700, 54454...",antigen,Demand: 4 - Capacity: base_capacity,0.106067
1,Hepatitis_B,"[386971200, 341644400, 323790700, 394259000, 3...",antigen,Demand: 4 - Capacity: base_capacity,0.106067
1,Hib,"[304354400, 308656200, 347768600, 332411700, 3...",antigen,Demand: 4 - Capacity: base_capacity,0.106067
1,Measles,"[429846900, 502939800, 572604400, 617561300, 6...",antigen,Demand: 4 - Capacity: base_capacity,0.106067
...,...,...,...,...,...
35,PT_Bio,"[46606392.98975441, 46293481.55039343, 4540880...",manufacturer,Demand: 4 - Capacity: other,0.003465
35,Panacea_Biotec,"[10767253.25728967, 10711528.39392783, 1106849...",manufacturer,Demand: 4 - Capacity: other,0.003465
35,Pfizer,"[85444251.2303147, 84318740.55897625, 83519126...",manufacturer,Demand: 4 - Capacity: other,0.003465


In [10]:
pair_df.loc[1]

unit                                                              HPV
values              [27340000, 35973200, 41902900, 49597700, 54454...
type                                                          antigen
pair_idx                                                            1
pair                              Demand: 4 - Capacity: base_capacity
pair_probability                                             0.106067
Name: 1, dtype: object

In [11]:
pair_df['unit'].unique()

array(['Diphtheria', 'HPV', 'Hepatitis_B', 'Hib', 'Measles', 'Mumps', 'PCV',
       'Pertussis', 'Polio', 'Rotavirus', 'Rubella', 'Tetanus', 'AJ_Vaccines',
       'BB_NCIPD', 'Bharat_Biotech', 'Bilthoven', 'Biological_E',
       'China_National', 'GSK', 'Haffkine_Bio', 'LG_Chem', 'Merck_Sharp',
       'PT_Bio', 'Panacea_Biotec', 'Pfizer', 'Sanofi', 'Serum_Institute'],
      dtype=object)

In [12]:
pair_df

,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[657354000, 751729500, 827191800, 881049800, 8...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
1,HPV,"[27340000, 35973200, 41902900, 49597700, 54454...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
2,Hepatitis_B,"[386971200, 341644400, 323790700, 394259000, 3...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
3,Hib,"[304354400, 308656200, 347768600, 332411700, 3...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
4,Measles,"[429846900, 502939800, 572604400, 617561300, 6...",antigen,1,Demand: 4 - Capacity: base_capacity,0.106067
...,...,...,...,...,...,...
940,PT_Bio,"[46606392.98975441, 46293481.55039343, 4540880...",manufacturer,35,Demand: 4 - Capacity: other,0.003465
941,Panacea_Biotec,"[10767253.25728967, 10711528.39392783, 1106849...",manufacturer,35,Demand: 4 - Capacity: other,0.003465
942,Pfizer,"[85444251.2303147, 84318740.55897625, 83519126...",manufacturer,35,Demand: 4 - Capacity: other,0.003465
943,Sanofi,"[88590563.9429312, 86836129.89446063, 82910332...",manufacturer,35,Demand: 4 - Capacity: other,0.003465


In [13]:
pair_df.groupby('unit').count()

,values,type,pair_idx,pair,pair_probability
unit,,,,,
AJ_Vaccines,35,35,35,35,35
BB_NCIPD,35,35,35,35,35
Bharat_Biotech,35,35,35,35,35
Bilthoven,35,35,35,35,35
Biological_E,35,35,35,35,35
China_National,35,35,35,35,35
Diphtheria,35,35,35,35,35
GSK,35,35,35,35,35
HPV,35,35,35,35,35
